# 07. Generalização em cópias sintéticas (CORD)

Este notebook avalia o modelo congelado da v2 em notas de outra origem (CORD, validation e test), com adulterações de copy-move geradas por um programa de parâmetros fixos. Nenhum componente é treinado, recalibrado ou ajustado: usam-se `results/modelo_v2.joblib` e o limiar de `results/configuracao_congelada.json`. Nada de `data/findit2/` nem de `data/variante_sroie_kaggle/` é lido.

**O que o experimento mede:** se o modelo marca como "Revisar" colagens exatas de uma caixa de texto em notas desconhecidas, e a taxa de alarme em notas intactas. **O que ele não mede:** adulterações reais, feitas por pessoas; a colagem sintética é a mais fácil de detectar para um método de correlação, pois copia pixels sem transformação.

Os números aparecem apenas como saída das células; este notebook não registra conclusões.

In [1]:
import sys
sys.path.append("../src")
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

from avaliacao_sintetica import carregar_amostras, localizacao, metricas_classificacao, pontuar
from conjunto_sintetico import montar_conjunto

RAIZ = Path("..")

## 1. Gerador congelado

O gerador (`src/gerar_copy_move_sintetico.py`) escolhe um componente conexo da máscara de Otsu com lados entre 1,0% e 6,0% da largura da imagem e o cola sobre o centro de outro componente de texto da mesma nota, sem sobreposição com a origem. Os pixels são copiados sem transformação. Os parâmetros e o hash SHA-256 do arquivo foram gravados em `results/gerador_congelado.json` antes de qualquer execução do pipeline. A célula confere que o arquivo atual ainda tem esse hash.

In [2]:
congelado = json.loads((RAIZ / "results/gerador_congelado.json").read_text(encoding="utf-8"))
hash_atual = hashlib.sha256((RAIZ / congelado["arquivo"]).read_bytes()).hexdigest()
print("hash congelado:", congelado["sha256"])
print("hash atual:    ", hash_atual)
print("idêntico:", hash_atual == congelado["sha256"])
print(json.dumps({k: congelado[k] for k in ("seed", "limites_caixa", "destino", "regras")}, ensure_ascii=False, indent=2))

hash congelado: 75bd3bfe3ee82a73db5623d5426cbd7f43abd4c852a544cc01a560a5c9c364d9
hash atual:     75bd3bfe3ee82a73db5623d5426cbd7f43abd4c852a544cc01a560a5c9c364d9
idêntico: True
{
  "seed": 42,
  "limites_caixa": {
    "lado_min_fracao_da_largura": 0.01,
    "lado_max_fracao_da_largura": 0.06,
    "area_tinta_min_origem_px": 20,
    "fracao_tinta_min_origem": 0.15
  },
  "destino": {
    "area_tinta_min_componente_px": 15,
    "fracao_tinta_min_caixa": 0.1,
    "tentativas": 200,
    "ponto": "centroide de outro componente de texto; caixa centrada nele"
  },
  "regras": [
    "sem sobreposição entre caixa de origem e de destino",
    "caixa de destino inteira dentro da imagem",
    "pixels colados copiados sem transformação",
    "máscara de Otsu como em features._text_mask"
  ]
}


## 2. Montagem do conjunto

As 200 notas de `data/cord/imagens/` são lidas em tons de cinza. Em cada uma tenta-se gerar a colagem (seed 42). Notas em que o gerador não encontra par origem/destino são fotos de fundo escuro ou texturizado, sem componente na faixa de tamanho; elas saem **das duas classes**, porque retirá-las só da classe adulterada tornaria as classes diferentes por construção. As notas restantes são divididas ao meio com seed 42, cada uma em uma só classe. Saída: `data/cord_sintetico/` (PNG e `inventario.csv`).

In [3]:
resumo = montar_conjunto(RAIZ / "data/cord/imagens", RAIZ / "data/cord_sintetico")
print({k: v for k, v in resumo.items() if k != "excluidas_sem_par"})
print("notas excluídas (gerador sem par):", len(resumo["excluidas_sem_par"]), resumo["excluidas_sem_par"])

{'n_adulteradas': 95, 'n_autenticas': 95, 'imagens_duplicadas': 0, 'fracao_texto_caixa_min': 0.1, 'fracao_texto_caixa_media': 0.34308104240962695}
notas excluídas (gerador sem par): 10 ['cord_test_0018.png', 'cord_test_0020.png', 'cord_test_0038.png', 'cord_test_0039.png', 'cord_test_0061.png', 'cord_validation_0009.png', 'cord_validation_0025.png', 'cord_validation_0045.png', 'cord_validation_0059.png', 'cord_validation_0068.png']


Verificações do conjunto: cada nota de origem aparece uma vez; nenhuma imagem é idêntica a outra (MD5 dos pixels, calculado na montagem); e a caixa adulterada contém texto da nota original (a fração de pixels de Otsu na caixa, antes da colagem, tem mínimo igual ao limite do gerador, 0,10), ou seja, não cobre só fundo.

In [4]:
amostras = carregar_amostras(RAIZ / "data/cord_sintetico")
inv = pd.read_csv(RAIZ / "data/cord_sintetico/inventario.csv")
print(inv["classe"].value_counts().to_dict())
print("notas de origem repetidas:", int(inv["nota_origem"].duplicated().sum()))
print("imagens idênticas:", resumo["imagens_duplicadas"])
print("fração de texto na caixa antes da colagem: mín = %.3f, média = %.3f" % (resumo["fracao_texto_caixa_min"], resumo["fracao_texto_caixa_media"]))

{'adulterada': 95, 'autentica': 95}
notas de origem repetidas: 0
imagens idênticas: 0
fração de texto na caixa antes da colagem: mín = 0.100, média = 0.343


## 3. Classificação no limiar congelado

As 12 features da v2 são extraídas por `extract_features` e `component_features`, as mesmas do lote original, e o modelo congelado devolve a probabilidade. A decisão usa o limiar congelado (0,49656). Reportam-se sensibilidade (adulteradas marcadas) e taxa de falso alarme (autênticas marcadas), ambas com intervalo de Wilson de 95%, e a AUC com intervalo de bootstrap de 95% (2000 reamostras, seed 42).

In [5]:
valores, prob, limiar = pontuar(amostras)
y = np.array([int(s.is_tampered) for s in amostras])
metricas = metricas_classificacao(y, prob, limiar)
for k, v in metricas.items():
    print(f"{k}: {v}")

limiar: 0.49656330908964424
n_adulteradas: 95
n_autenticas: 95
sensibilidade: 0.3263157894736842
sensibilidade_acertos: 31
sensibilidade_wilson: (0.2403857202573143, 0.4257467583751644)
falso_alarme: 0.2631578947368421
falso_alarme_marcadas: 25
falso_alarme_wilson: (0.18506401390488098, 0.3596620933212263)
auc: 0.487202216066482
auc_ic_bootstrap: (0.4039943616029823, 0.5704187210955817)


## 4. Histórico JPEG nas duas classes

O CORD é distribuído em PNG, e a feature `has_jpeg_history` pode separar as classes por artefato do conjunto, não por adulteração (viés semelhante ao RGBA no *Find it again!*). A célula compara a taxa dessa feature nas duas classes, com Wilson de 95%.

In [6]:
from evaluation import wilson_score_interval

historico = np.array([v["has_jpeg_history"] for v in valores]).astype(bool)
taxas = {}
for nome, mascara in (("adulterada", y == 1), ("autentica", y == 0)):
    n, k = int(mascara.sum()), int(historico[mascara].sum())
    taxas[nome] = {"n": n, "com_historico": k, "taxa": k / n, "wilson": wilson_score_interval(k / n, n)}
    print(nome, taxas[nome])
print("taxas diferentes:", taxas["adulterada"]["taxa"] != taxas["autentica"]["taxa"])

adulterada {'n': 95, 'com_historico': 9, 'taxa': 0.09473684210526316, 'wilson': (0.05064366025472933, 0.17033212322105415)}
autentica {'n': 95, 'com_historico': 8, 'taxa': 0.08421052631578947, 'wilson': (0.04328759724614253, 0.1574537909952457)}
taxas diferentes: True


## 5. Localização contra a caixa registrada

Em cada nota adulterada, mede-se se a técnica aponta a caixa colada melhor do que aponta pseudo-regiões ao acaso, pelo mesmo protocolo do projeto (`evaluate_region_technique`, 20 sorteios por imagem, mínimo de 10 válidos, seed 42). As pseudo-regiões têm o tamanho da caixa real, contêm texto (Otsu) e não sobrepõem a região anotada. Três medidas:

- **ELA:** acerto quando a média do erro na caixa supera a de um controle com texto.
- **Copy-move com caixa conhecida:** acerto quando o pico de correlação da caixa com o resto da imagem é pelo menos 0,979 (limiar escolhido no `val` do *Find it again!*). Usa a caixa registrada, então mede a detecção dada a localização.
- **Copy-move por busca cega:** acerto quando alguma caixa detectada sem gabarito (limiar 0,99) cobre pelo menos 50% da região.

In [7]:
loc = localizacao([s for s in amostras if s.is_tampered])
for nome, r in loc.items():
    print(nome, {k: r[k] for k in ("n_samples", "real_hits", "real_hit_rate", "real_wilson", "chance_hit_rate", "chance_hit_std", "chance_wilson", "exclusions")})

ela {'n_samples': 95, 'real_hits': 85, 'real_hit_rate': 0.8947368421052632, 'real_wilson': (0.8169777184937429, 0.9418121027960785), 'chance_hit_rate': 0.45, 'chance_hit_std': 0.1557411928454902, 'chance_wilson': (0.42775319794623956, 0.47244858354650665), 'exclusions': {'text_mask_failed': 0, 'real_region_failed': 0, 'few_valid_pseudo': 0}}
copy_move_caixa_conhecida {'n_samples': 93, 'real_hits': 93, 'real_hit_rate': 1.0, 'real_wilson': (0.9603310973796383, 0.9999999999999999), 'chance_hit_rate': 0.11691895773589572, 'chance_hit_std': 0.1199744284739346, 'chance_wilson': (0.10189081760090332, 0.13105335627026238), 'exclusions': {'text_mask_failed': 0, 'real_region_failed': 1, 'few_valid_pseudo': 1}}
copy_move_busca_cega {'n_samples': 95, 'real_hits': 1, 'real_hit_rate': 0.010526315789473684, 'real_wilson': (0.0018605283639709959, 0.05724009323665061), 'chance_hit_rate': 0.0, 'chance_hit_std': 0.0, 'chance_wilson': (-2.168404344971009e-19, 0.002017814927460352), 'exclusions': {'text_ma

## 6. Registro dos resultados

Os números das células anteriores são gravados em `results/generalizacao_sintetica.json`, sem texto interpretativo.

In [8]:
saida = {"modelo": "results/modelo_v2.joblib", "limiar": limiar, "gerador_sha256": hash_atual,
         "conjunto": {k: v for k, v in resumo.items()}, "classificacao": metricas,
         "has_jpeg_history": taxas, "localizacao": loc}
(RAIZ / "results/generalizacao_sintetica.json").write_text(json.dumps(saida, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print("gravado")

gravado
